# Koala-Kangaroo YOLO

## Logging And Setup

We setup our logging using `wandb`
We also initialize our global variables

In [ ]:
from ultralytics import settings

settings.update({"wandb": True,
                 "clearml": False,
                 "comet": False})
YOLO_MODEL = "yolo11s.pt"
PROJECT_NAME = "koala-kangaroo"
EXPERIMENT_NAME = "exp1"
DATASET_PATH = "data/yolo_koala_kangaroo.v1-original.yolov11/data.yaml"
BEST_MODEL_PATH = f"{PROJECT_NAME}/{EXPERIMENT_NAME}/weights/best.pt"

## Training

We specify the training path for our dataset.

For batch size, given we are training in our laptop and faced GPU out of memory problem, we leave it to Yolo to auto batch using -1 which seemed to solve the memory problem.


In [ ]:
!ls -la data/yolo_koala_kangaroo.v1-original.yolov11/train/images | wc -l

In [ ]:
from ultralytics import YOLO
from ultralytics import settings

model = YOLO(YOLO_MODEL)  # Load a pre-trained YOLO model
result = model.train(data=DATASET_PATH,
                     epochs=20, # number of training epochs
                     save_period=1, # save every epoch
                     batch=-1, # auto batch size, previously set to 16,64 and caused us to crash for GPU memory reasons
                     device=0, # use GPU 0
                     project=PROJECT_NAME, # set project name  for logging in wandb
                     name=EXPERIMENT_NAME, # set experiment name for logging in wandb
                     plots=True)

## Validation

Here we are looking for mAP50 and mAP50-95 score

In [ ]:
model = YOLO(BEST_MODEL_PATH)
validation_results = model.val(data=DATASET_PATH, device="0")


## Export

In [ ]:
model = YOLO(BEST_MODEL_PATH)
exported_path = model.export(format="openvino", int8=True)

## Inference

In [ ]:
import ultralytics
from ultralytics import YOLO
from PIL import Image

source = 'test_image.jpg'
model = YOLO(BEST_MODEL_PATH, task='detect')
result = model(source, conf=0.5, iou=0.6)

# Visualize the results
for i, r in enumerate(result):
    print(r)
    # Plot results image
    im_bgr = r.plot()  # BGR-order numpy array
    im_rgb = Image.fromarray(im_bgr[..., ::-1])  # RGB-order PIL image

    # Show results to screen (in supported environments)
    r.show()

    # Save results to disk
    r.save(filename=f"results{i}.jpg")